In [1]:
from locale import normalize

import pandas as pd
import numpy as np
import  matplotlib.pyplot as plt
import  seaborn as sns

df = pd.read_csv("../data/raw/ai4i2020.csv")

In [4]:
# Separate inputer features and target

X = df.drop(
    columns=[
        "UDI",
        "Product ID",
        "Machine failure",
        "TWF",
        "HDF",
        "PWF",
        "OSF",
        "RNF"
    ]
)

y = df["Machine failure"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

print("\nSelected features:")
print(X.columns.tolist())

Feature shape: (10000, 6)
Target shape: (10000,)

Selected features:
['Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']


In [7]:
# Inspecting categories

print(X["Type"].value_counts())
print(X["Type"].unique())

Type
L    6000
M    2997
H    1003
Name: count, dtype: int64
<StringArray>
['M', 'L', 'H']
Length: 3, dtype: str


In [10]:
X_encoded = pd.get_dummies(
    X, columns=["Type"],
    dtype=int
)

print(X_encoded.head())
print(X_encoded.shape)
print(X_encoded.columns.tolist())

   Air temperature [K]  Process temperature [K]  Rotational speed [rpm]  \
0                298.1                    308.6                    1551   
1                298.2                    308.7                    1408   
2                298.1                    308.5                    1498   
3                298.2                    308.6                    1433   
4                298.2                    308.7                    1408   

   Torque [Nm]  Tool wear [min]  Type_H  Type_L  Type_M  
0         42.8                0       0       0       1  
1         46.3                3       0       1       0  
2         49.4                5       0       1       0  
3         39.5                7       0       1       0  
4         40.0                9       0       1       0  
(10000, 8)
['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Type_H', 'Type_L', 'Type_M']


In [15]:
# Train, Test splitting
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_encoded,
                                                    y,
                                                    test_size=0.2,
                                                    random_state=42,
                                                    stratify=y)

print("Training features:", X_train.shape)
print("Testing features:", y_train)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True)*100)
print(y_test.value_counts(normalize=True)*100)

Training features: (8000, 8)
Testing features: 4058    0
1221    0
6895    0
9863    0
8711    0
       ..
980     0
4266    0
7772    0
5780    0
1424    0
Name: Machine failure, Length: 8000, dtype: int64

Training target distribution:
Machine failure
0    96.6125
1     3.3875
Name: proportion, dtype: float64
Machine failure
0    96.6
1     3.4
Name: proportion, dtype: float64


In [18]:
from sklearn.preprocessing import  StandardScaler

numerical_features = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
]

scaler = StandardScaler()

X_trained_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_trained_scaled[numerical_features] = scaler.fit_transform(X_train[numerical_features])

X_test_scaled[numerical_features] = scaler.transform(X_test[numerical_features])

print(X_trained_scaled[numerical_features].describe())

       Air temperature [K]  Process temperature [K]  Rotational speed [rpm]  \
count         8.000000e+03             8.000000e+03            8.000000e+03   
mean          1.758593e-15             1.053468e-14            3.010925e-16   
std           1.000063e+00             1.000063e+00            1.000063e+00   
min          -2.356591e+00            -2.910801e+00           -2.052017e+00   
25%          -8.541261e-01            -8.152711e-01           -6.484822e-01   
50%           4.735268e-02             6.349964e-02           -2.008982e-01   
75%           7.485028e-01             7.394771e-01            4.069319e-01   
max           2.250967e+00             2.564616e+00            7.441184e+00   

        Torque [Nm]  Tool wear [min]  
count  8.000000e+03     8.000000e+03  
mean  -1.474376e-16    -3.064216e-17  
std    1.000063e+00     1.000063e+00  
min   -3.613500e+00    -1.692947e+00  
25%   -6.790515e-01    -8.597185e-01  
50%    9.645501e-03    -1.076908e-02  
75%    6.783803